<a href="https://colab.research.google.com/github/lion-of-naples/Generative-AI-Overview/blob/main/Public_Facing_GenerativeAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generative AI Overview

This notebook is a living document that will be maintained by a committee of experts with the goal of providing a digestible high-level overview of the current state of Generative AI.

*   As breakthroughs happen, we will document them and provide breakouts to more detailed notebooks and other resources.
*   We envision multiple focus areas of interest, where alumni and external members can join, dig deeper into that area, and then take over (admin privileges for the notebook), and contribute back to this high-level notebook.
*   We will have signup sheets for each area available at the end of this event.

---

**Table of contents**
*   [Generative AI in three parts](#generative-ai-in-three-parts) — Pre-training, Post-training (SFT), Post-training (RL)
*   [How we got here](#how-we-got-here) — a short history of language models, from Shannon to open models
*   [Pretraining](#pretraining) — Datasets and data preparation, encoding and byte-level models, the transformer and how it has changed, [scaling laws](#scaling-laws), inference
*   [Post-training: Supervised fine tuning](#post-training---supervised-fine-tuning)
*   [Post-training: Reinforcement learning](#post-training---reinforcement-learning)
*   [Evaluating language models](#evaluating-language-models) — internal versus external metrics, perplexity, contamination
*   [Quantum Computing and Generative AI](#quantum-computing-and-generative-ai): hardware evidence, technique maturity, and scoped security deadlines
*   [Further learning](#further-learning) — courses, books, and open projects
*   [Glossary](#glossary)

> **New in October 2026:** a short history of language models, pretraining data preparation, byte-level models, Transformer changes since 2017, scaling laws, faster inference, named reinforcement-learning methods, an evaluation section, and further learning, drawing on Stanford CS336 (Spring 2026) and the original papers.

> **New in 2026:** quantum content appears in two forms — short **Quantum Angle** callouts inside the pipeline sections (pretraining, encoding, training, inference), and a standalone [Quantum Computing and Generative AI](#quantum-computing-and-generative-ai) section before the glossary.

**Navigation:** the inline section links are verified in exported HTML. GitHub's notebook preview can rewrite fragment links, and Colab does not reliably scroll to these custom anchors; in Colab, use its built-in **Table of contents** panel to jump between sections.

## Perplexity.ai as a Resource
We will use Perplexity as our main research tool in this notebook. Here are a few of the reasons why:

*   Uses the power of GenAI models for searching
*   Lists references with returned results
*   Two of its four co-founders, Aravind Srinivas and Andy Konwinski, earned PhDs at UC Berkeley
*   Will use as the primary source (of multiple sources) of truth to reduce subjectivity

**Sign up for Perplexity:**

*   [Perplexity Signup](https://perplexity.ai)


**Here is what Google says about Perplexity.ai**

![perplexity](https://drive.google.com/uc?export=view&id=1z-SjOnAySblqKFOG1d6UN_tzlwYNQUYq)



## Is Generative AI a Synonym for AI?


The short and scientific answer is no although some will disagree. Here is a screenshot showing the result of asking this question in **Perplexity.ai**:

![genAI](https://drive.google.com/uc?export=view&id=131Fgt3PlEYETBLiYut5opJ8Koy4kzVIQ)


<a id="generative-ai-in-three-parts"></a>

## Generative AI in three parts
The composition of modern Generative AI can be broken down into three phases:
1.   **Pre-training** — Downloading the Internet and other large datasets (Analogy: grade school level background knowledge)
2.   **Post-training: Supervised Fine Tuning (SFT)** — A type of transfer learning that updates pre-training based on task-specific data (Analogy: college with a major)
3.   **Post-training: Reinforcement Learning** — We break this into two types:
   - **Human-based Reinforcement Learning** — Uses humans (and now other Large Language Models (LLMs)) to vote on the correct prompt and answer pairs for GenAI chats so models learn from human experience (Analogy: supervised on-the-job training).
   - **Full Reinforcement Learning** — Lets models learn to optimize their responses through experience-based learning that is not human- or LLM-coded. Models are given an environment with desired outcomes or goals but multiple paths or actions to reach them (Analogy: non-supervised on-the-job training).

The remainder of this notebook will be broken up into these three areas. Each of these sections will become focus groups where members can meet, discuss breakthroughs and the potential for breakthroughs, and contribute code for the group.

<a id="how-we-got-here"></a>

## How We Got Here — A Short History of Language Models

Reviewed October 13, 2026. This follows the history Percy Liang gave in Lecture 1 of Stanford's CS336, *Language Modeling from Scratch*, Spring 2026 ([CS336 course site](https://cs336.stanford.edu/)), checked against the original papers.

Language models are older than deep learning. Claude Shannon used them in 1951 to estimate how predictable English is ([Prediction and Entropy of Printed English](https://www.princeton.edu/~wbialek/rome/refs/shannon_51.pdf)). For decades afterward, n-gram models, which predict the next word from the previous few, were an important part of machine translation and speech recognition systems, keeping their output fluent.

| Year | Milestone | Why it mattered |
|---|---|---|
| 2003 | [A Neural Probabilistic Language Model](https://www.jmlr.org/papers/v3/bengio03a.html) — Bengio, Ducharme, Vincent, and Jauvin | A neural network learned a distributed representation for words from a short window of context |
| 2014 | [Sequence to Sequence Learning](https://arxiv.org/abs/1409.3215) — Sutskever, Vinyals, and Le | Compressed a whole input sentence into one vector, then generated the output from it |
| 2014 | [Attention for translation](https://arxiv.org/abs/1409.0473) — Bahdanau, Cho, and Bengio; [Adam](https://arxiv.org/abs/1412.6980) — Kingma and Ba | Attention let the model look back at the source instead of relying on one vector; Adam, and later AdamW, became the standard optimizer |
| 2017 | [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — Vaswani et al.; [sparse mixture of experts](https://arxiv.org/abs/1701.06538) — Shazeer et al. | The Transformer architecture, and a way to grow capacity without growing the compute per token |
| 2018 | [ELMo](https://arxiv.org/abs/1802.05365), [BERT](https://arxiv.org/abs/1810.04805), [GPT](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) | Pretrain on large amounts of text, then fine-tune for a downstream task |
| 2019 | [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf); [T5](https://arxiv.org/abs/1910.10683) | Byte-level BPE plus scale; T5 converted every task into text in, text out |
| 2020 | [Scaling laws](https://arxiv.org/abs/2001.08361) — Kaplan et al.; [GPT-3](https://arxiv.org/abs/2005.14165) | Loss became predictable from scale; a 175-billion-parameter model learned tasks from examples in the prompt |
| 2021–2022 | [Gopher](https://arxiv.org/abs/2112.11446), [PaLM](https://arxiv.org/abs/2204.02311), [Chinchilla](https://arxiv.org/abs/2203.15556) | Chinchilla showed that the large models of the time were significantly undertrained |
| 2020–2023 | [The Pile](https://arxiv.org/abs/2101.00027) (EleutherAI), [OPT-175B](https://arxiv.org/abs/2205.01068), [BLOOM](https://arxiv.org/abs/2211.05100), [LLaMA](https://arxiv.org/abs/2302.13971) | Open replications after GPT-3; early ones were weaker, and Meta's LLaMA series went on to lead the open-weight ecosystem |
| 2023 | [GPT-4 Technical Report](https://arxiv.org/abs/2303.08774) | Frontier labs stopped publishing how their models are built (quoted below) |

The GPT-4 report states: "Given both the competitive landscape and the safety implications of large-scale models like GPT-4, this report contains no further details about the architecture (including model size), hardware, training compute, dataset construction, training method, or similar" ([GPT-4 Technical Report](https://arxiv.org/abs/2303.08774)).

**Four eras of use.** Liang describes how the idea of a language model has changed: first something you fine-tune (the BERT era), then something you prompt (GPT-3), then something you talk to (ChatGPT), and now agents that carry out long, multi-step tasks. The fundamentals have changed much less: GPUs and kernels, gradient-based optimization, and the Transformer with attention.

**Open-weight versus fully open.** Open-weight models, including Meta's Llama series, Mistral, DeepSeek, and Qwen, release their trained weights; in Liang's assessment they are approaching closed models, depending on who benchmarks them and how. A smaller group goes further and releases the data and code as well. AI2's OLMo releases "open training data and training and evaluation code" ([OLMo paper](https://arxiv.org/abs/2402.00838)), the lecture also names NVIDIA, and Liang's own Marin project develops models in the open, with experiments preregistered and visible as they run ([Marin](https://marin.community/)). Even the open-weight papers usually omit details such as the data mixture, so they cannot be fully reproduced.

<a id="pretraining"></a>

# Pretraining
The first step in building a Generative AI platform is to conduct a pretraining step consisting of training models on a large and diverse set of quality data. **This is typically the most expensive and time-consuming part of training a large language model (LLM)** — the data is often many terabytes and requires significant hardware infrastructure to train.

### Quantum Angle — Does Quantum Computing Make Pretraining Cheaper?

Quantum callout: one of four short inserts examining possible connections to this pipeline. Evidence reviewed October 3, 2026; this is not a forecast of when quantum advantage will arrive.

**The sources reviewed here do not establish a practical, end-to-end quantum advantage for frontier LLM pretraining.** In particular, algorithmic speedup claims must include data access, state preparation, measurement, error correction, and a competitive classical baseline; a digital-health QML review illustrates how encoding assumptions complicate comparisons, but does not itself evaluate frontier LLM training ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z)).

*   **Data-loading assumptions matter.** Generic amplitude preparation has cost linear in the number of input values; structured inputs or special access assumptions can change that cost, while robust quantum random-access memory remains a hardware challenge ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z)).
*   **Physical-qubit counts are not a universal scale comparison.** IBM reported 120 physical qubits and 218 couplers for Nighthawk in November 2025, while QuEra lists a 256-atom analog machine; these are different architectures and capabilities, not interchangeable computational resources ([IBM](https://newsroom.ibm.com/2025-11-12-ibm-delivers-new-quantum-processors,-software,-and-algorithm-breakthroughs-on-path-to-advantage-and-fault-tolerance); [QuEra](https://www.quera.com/ai-info-page)).

**Evaluation rule for this notebook:** do not infer a useful transformer-training speedup from a quantum subroutine alone. Ask for the complete workload, assumptions, resource budget, and measured comparison.

### Pretraining Datasets — Fineweb Focus
![Fineweb pipeline on Hugging Face](https://drive.google.com/uc?export=view&id=1ul6WO2JL1H0B8BvO-TJq7z4l-8BhBHAM)
A great way to understand pretraining is by reading about and evaluating the Fineweb dataset offered by Hugging Face. This dataset is openly released under the ODC-By 1.0 license, and its use is also subject to Common Crawl's Terms of Use ([FineWeb dataset card](https://huggingface.co/datasets/HuggingFaceFW/fineweb)). Not all pretraining is open (e.g., OpenAI). Many open pre-training datasets start with [Common Crawl data](https://commoncrawl.org/), a free resource that is in essence a download of the Internet.

**Link to Fineweb**
*   [Fineweb on Hugging Face](https://huggingface.co/spaces/HuggingFaceFW/blogpost-fineweb-v1)
   
**Additional Datasets**
*   [Fineweb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu)
*   [RefinedWeb](https://arxiv.org/abs/2306.01116)
*   [Dolma](https://allenai.github.io/dolma/)
*   [C4 (Colossal Clean Crawled Corpus)](https://www.tensorflow.org/datasets/catalog/c4)
*   [Colossal-OSCAR](https://oscar-project.org/)


**Signup page for Pretraining Focus Group**
*   [Member Access to Page](https://docs.google.com/document/d/1rcwlegMf5PMm9hZqV3r7uSQ9-Qedg3IIsEGgpyj82I0/edit?usp=sharing)

**Output of Current Step:** The output of this step is a stream of filtered and raw text.

**Next Step:** Next we have to encode the data in machine readable format and add context.

### Beyond Fineweb — How Pretraining Data Is Made

Reviewed October 13, 2026, drawing on Lecture 1 of Stanford CS336 ([course site](https://cs336.stanford.edu/)) and the papers linked below.

In Liang's words, data "does not just fall from the sky." It has to be actively curated, and what goes into the training set largely decides what the model can do: if you want a model that speaks many languages, holds a conversation, or runs long coding tasks, the data has to reflect that.

**Where the data comes from.** Web pages from crawls, books, arXiv papers, GitHub code, and more. The Pile shows how varied a pretraining corpus is: 825 GiB of English text built from 22 different subsets, many from academic or professional sources ([The Pile](https://arxiv.org/abs/2101.00027)).

**The processing pipeline**

*   **Transformation:** most raw data is not text. It arrives as HTML, PDFs, or whole code directories and must be converted into usable text.
*   **Filtering:** keep the good material. A random Common Crawl document is often not worth training on. The FineWeb paper describes 15 trillion tokens from 96 Common Crawl snapshots, and the team tested its filtering choices by training models on the results ([FineWeb paper](https://arxiv.org/abs/2406.17557)); the dataset has since grown to more than 18.5 trillion tokens ([FineWeb dataset card](https://huggingface.co/datasets/HuggingFaceFW/fineweb)).
*   **Deduplication:** existing datasets contain many near-duplicates. One study removed a single 61-word English sentence that appeared more than 60,000 times in C4 ([Lee et al.](https://arxiv.org/abs/2107.06499)).
*   **Mixing:** deciding how much weight each source gets in the final training mix.
*   **Synthetic data:** rewriting real data into more useful forms. Prompting a model to rephrase web documents "like Wikipedia" or as questions and answers sped up pretraining on C4 by about three times ([Rephrasing the Web](https://arxiv.org/abs/2401.16380)).

**Three training stages, three kinds of data**

*   **Pretraining:** the large, broad corpus described above.
*   **Mid-training:** smaller, higher-quality data placed near the end of pretraining, including long-context material such as large code repositories and books. OLMo 2's Dolmino Mix 1124, introduced through late-stage curriculum training, is a documented example ([OLMo 2](https://arxiv.org/abs/2501.00656)).
*   **Post-training:** conversations, and increasingly agent traces that include tool calls.

**Legal questions are part of the work.** Whether training on copyrighted data is fair use is contested, and some data has to be licensed. Code is a concrete case: GitHub's documentation says that without a license, "the default copyright laws apply," and no one may reproduce, distribute, or create derivative works ([GitHub Docs](https://docs.github.com/en/repositories/managing-your-repositorys-settings-and-features/customizing-your-repository/licensing-a-repository)). A data team has to decide whether to treat unlicensed code as permissive or to exclude it.

**The efficiency lens.** Liang frames filtering as an efficiency decision. Even if bad data does not directly harm the model, every hour of a fixed compute budget spent on redundant or low-quality text is an hour not spent on good text.

## Encoding Pretrained Data
The output of the pretraining data filtering step is raw text. Now that we have (basically) downloaded the Internet and have raw text, we need to encode that data so it can be input into a neural network and statistical patterns can be discovered.

### Encoding — Basics and Links to More Information
Encoding involves representing data in a way that neural networks can understand. Current LLM architectures rely on tokens for representation:

*   At the lowest level of computer encoding, symbols/tokens have a vocabulary of 1's and 0's (two possible symbols)
*   Grouping bits into bytes (8 bits each) shrinks the sequence length by 8 times and gives a vocabulary of 256 possible symbols, because 8 bits have 2⁸ = 256 combinations.
*   The byte-pair encoding (BPE) algorithm starts from single bytes, finds the most frequent pair of adjacent tokens in the training text, and merges that pair into one new token. Repeating this merge step many times grows the vocabulary one token at a time and shortens the sequences.
*   A larger vocabulary compresses text into fewer tokens, so more text fits in the context window and the model processes it faster. The trade-off is not lost information, because byte-level BPE is fully reversible; it is that rare tokens get fewer training examples and the embedding and output layers grow with the vocabulary.
*   For reference, OpenAI's GPT-4 tokenizer (`cl100k_base`) has about 100,000 tokens, and the tokenizer used by GPT-4o and later models such as o1 (`o200k_base`) has about 200,000 ([tiktoken](https://github.com/openai/tiktoken/blob/main/tiktoken/model.py)).
*   Converting raw text (bytes) into a sequence of tokens from a vocabulary like this is called tokenization.

### Quantum Angle — Encoding Classical Data into Qubits

Quantum callout: BPE tokenization and quantum state preparation are different operations. The comparison below is a teaching analogy about representing inputs, not a claim that quantum encoding replaces tokenization; the quantum preparation costs depend on the representation and access model ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z)).

| | Classical GenAI encoding | Quantum encoding |
|---|---|---|
| Representation | Token IDs from a vocabulary | Features encoded in basis states, rotation angles, or amplitudes ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z)) |
| Cost accounting | Classical preprocessing can be cached; model computation is separate | Include state preparation and repetitions required for measurement; costs depend on the encoding and access assumptions ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z)) |
| Scaling | Token sequence length depends on text and tokenizer | Simple parallel angle encoding can use constant depth with qubits linear in features; generic amplitude encoding trades logarithmic qubits for linear preparation cost ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z)) |

*   Amplitude encoding represents a normalized vector of 2ⁿ amplitudes in n qubits, but that compact representation does not grant free loading or full classical readout of the vector ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z)).
*   When dimensionality reduction is used, compare quantum and classical models on equivalent inputs and also report a strong classical baseline using the full data; the reviewed digital-health literature does not establish a consistent empirical quantum advantage ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z)).

**Takeaway:** compare end-to-end costs under explicit assumptions. Neither compact quantum representation nor cached classical preprocessing is, by itself, evidence of a learning advantage.

#### Basic Encoding Example
To illustrate how encoding works in Generative AI we use one webpage that was crawled by Common Crawl, sent through the Fineweb pipeline to be filtered and encoded, and used as training input for LLMs. Expand the cells below to see the raw text, binary equivalent, and different encoding methods.

##### Screenshot from Hugging Face
This screenshot shows how the webpage is represented on Hugging Face.
![Webpage as represented on Hugging Face](https://drive.google.com/uc?export=view&id=1FdJV1B42KY0c63_vQRxhNPlTTwckHCy0)

##### Screenshot from source for Hugging Face
This screenshot shows the webpage at its source. Note the links and some words that were filtered out in the Fineweb pipeline.
![Webpage source before Fineweb filtering](https://drive.google.com/uc?export=view&id=1LYINPviJTxkHBTx217yEYJLLavmhGndK)

##### Text From Example
*Optional read — full example text (494 words):*


---


We're not going to lie, we were crapping our pants when BDizzle pulled that hammy 5 minutes into the game. Not to mention his knee was killing him. But something about this team and these fans said "we ain't losing." All of us were hoping Davis could pull a Willis Reed, but he did way more than that. He truly had one of the all time epic playoff performances in this series and in this game. And you know what, so did the Warriors fans. For those three home games, it was INSANE at the Arena.
Meanwhile Stephen "Dead Eye" Jackson was the man, hitting 7 threes and going for 33. J-Rich, Barnes and Biedrins were also clutch as hell. If you had told us before going into this game that the Warriors would absolutely blow out the Mavs even though Baron Davis pulled his hammy shortly after tipoff, we would have said you're nuts.
But that 18-0 run in the third quarter absolutely killed the Mavs. They never recovered, you could see it on their faces. Meanwhile, Nowitzki was beyond awful, going for only 8 pts. I think we need to reroute that MVP trophy immediately to Baron Davis.
We've also gotta tip our hat to Nellie, who totally outcoached Avery, and got his extremely sweet revenge on Cuban. We knew we had the Mavs from the first game when Avery switched up his starting lineup.
Best of all about this Game 6, there will be no ref conspiracy theories. The Mavericks simply got CRUSHED, and that's all she wrote.
Now, it's time for us to call out all of those who called us crazy (and much worse things) for picking the Warriors to win this series. I was a bit off, since I said the Warriors would win in 7. Meanwhile Dwyermaker revised his in 7 pick to 6 early in the series. We're also going to send a huge shout out to Dan Shanoff who also picked the Warriors to win this series as well.
Below are some of the best insults people threw at us for picking the Warriors before the series began.
"The only way the Warriors will win, 100% injury rate."
"are you high?"
"Nellie's teams have NEVER proved to be effective in the playoffs against quality competition."
"LMAO, you're too delusional."
"Have you ever watched the Mavs outside of GS games?"
"Wow. You have got to be the most ignorant basketball fan in the history of the game.Golden State doesn't play defense, they've never been in the playoffs, and we ALL know that how you play leading up to the playoffs doesn't mean squat once you're in."
"Mavs in 6 at worst." (It took out the next comment - Your’e fat)
We appreciate the comments guys. But look who's laughing now. We are. Enjoy your summer Mavs fans. I'm sure that MVP trophy that Herr Luftball gets will help ease the pain. Oh wait, no it won't.

##### ASCII/UTF-8 Binary from Example
*Optional read — full binary representation.* The same passage is 2,669 bytes in UTF-8, or 21,352 binary characters (8 per byte). Without tokenization, each consecutive 1 and 0 would need to be encoded and run through the neural network; current space and compute make that infeasible (no compression).


---


...

##### Tokenization for GPT-4
**Tiktokenizer screenshot of all text input into GPT-4 tokenizer**
![GPT-4 tokenizer in Tiktokenizer](https://drive.google.com/uc?export=view&id=1IeyKbTJwXB6vyz06Jk8C8g0ug71pnjM0)

##### Tokenization for o1
o1 uses the same `o200k_base` tokenizer as GPT-4o ([tiktoken](https://github.com/openai/tiktoken/blob/main/tiktoken/model.py)).
![o1 tokenizer](https://drive.google.com/uc?export=view&id=1nfhEK4AUMc-7biOHrK88m5jDbSwwwOur)

##### Difference between tokenizers

**Encoding for GPT-4**
![Encoding for GPT-4](https://drive.google.com/uc?export=view&id=15JPhCtUi8h5dMlHV51irzX7VYLcHWmmN)



---




**Encoding for GPT-4o**
![Encoding for GPT-4o](https://drive.google.com/uc?export=view&id=1yNGUvvHTn8rg7rsArE5Re7BMEJcRqKvw)

<a id="beyond-tokenizers-byte-level-models"></a>

### Beyond Tokenizers — Byte-Level Models

Research note reviewed October 13, 2026. It draws on Lecture 1 (overview and tokenization) of Stanford's CS336, *Language Modeling from Scratch*, Spring 2026, taught by Percy Liang ([CS336 course site](https://cs336.stanford.edu/)), checked against the primary papers linked below.

Tokenizers are a practical compromise, not a law of nature. A tokenizer converts raw bytes into a sequence of integer IDs and must convert them back exactly; if encoding and then decoding does not return the original text, the tokenizer is broken. In the lecture, Liang said he hopes each year that he will no longer need to teach tokenization, because the goal is a model that works directly on bytes end to end. As of the Spring 2026 lecture, he described byte-level approaches as promising but not yet scaled to frontier models, which still use tokenizers.

**Why tokenizers exist: the three simpler options all fall short**

*   **Characters:** Unicode has a very large symbol set, and many characters are rare. The GPT-2 paper notes that a code-point vocabulary would need over 130,000 base symbols before any merges ([GPT-2 paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)).
*   **Raw bytes:** only 256 possible symbols, but one token per byte makes sequences long. Because standard attention compares every position with every other, cost grows with the square of sequence length.
*   **Whole words:** each token is meaningful, but the vocabulary is effectively unbounded, and words never seen in training fall back to an "unknown" token.
*   **Byte-pair encoding (BPE):** the practical middle ground. It began as a data-compression algorithm ([Gage, 1994](https://www.derczynski.com/papers/archive/BPE_Gage.pdf)), was adapted for neural machine translation ([Sennrich, Haddow, and Birch, ACL 2016](https://arxiv.org/abs/1508.07909)), and was applied directly to bytes in GPT-2, which needs a base vocabulary of only 256 and expands it to 50,257 tokens ([GPT-2 paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)). Training repeatedly merges the most frequent adjacent pair into a new token, so common sequences become one token and rare ones stay split into smaller pieces.

**Measured on this notebook's example**

The 494-word Fineweb example above is 2,669 bytes in UTF-8. Measured with OpenAI's `tiktoken` library (version 0.14.0) on October 13, 2026:

| Tokenizer | Used by | Vocabulary | Tokens | Bytes per token | Round trip |
|---|---|---|---|---|---|
| `cl100k_base` | GPT-4 | about 100,000 | 634 | 4.21 | exact |
| `o200k_base` | GPT-4o, o1 | about 200,000 | 616 | 4.33 | exact |

Doubling the vocabulary shortened this text by about 3%. Bytes per token is the compression ratio: higher means shorter sequences, but a larger vocabulary also means more tokens that each appear rarely in training.

**Tokenizer quirks worth knowing** (`o200k_base`, same date)

*   `hello` and ` hello` (with a leading space) are different tokens, IDs 24912 and 40617, with no built-in relationship between them.
*   `2026` becomes two tokens, `202` and `6`, while `123456789` becomes `123`, `456`, and `789`. How numbers split affects arithmetic, and it varies by tokenizer.

**The byte-level research direction**

| Work | First posted | Core idea | Reported result (authors' claims) |
|---|---|---|---|
| [ByT5](https://arxiv.org/abs/2105.13626) — Xue et al. | May 2021; TACL 2022 | A standard Transformer with minimal changes, trained directly on bytes | Competitive with token-level models; more robust to noise and better on spelling- and pronunciation-sensitive tasks |
| [MEGABYTE](https://arxiv.org/abs/2305.07185) — Yu et al. | May 2023 | Split bytes into fixed patches; a local model works within each patch and a global model across patches | Sequences over one million bytes; competitive with subword models on long-context language modeling |
| [Byte Latent Transformer](https://arxiv.org/abs/2412.09871) — Pagnoni et al. | December 2024 | Dynamically sized patches, segmented by how hard the next byte is to predict | First FLOP-controlled scaling study of byte-level models, up to 8B parameters and 4T training bytes; matches tokenization-based models at scale |
| [H-Net](https://arxiv.org/abs/2507.07955) — Hwang, Wang, and Gu | July 2025 | Learns its own chunking jointly with the model, replacing the tokenize–model–detokenize pipeline | At matched compute and data, a byte-level H-Net outperforms a BPE Transformer; larger gains on Chinese, code, and DNA |

**What any replacement has to do**

Liang's lecture names two properties any end-to-end approach still needs. First, the model must operate on abstractions above raw units; this matters even more for video or DNA, where individual bytes carry little signal. Second, chunks should be variable: predictable stretches should cost fewer units and rare or surprising stretches more. Byte Latent Transformer's entropy-based patches and H-Net's learned chunking are both attempts to meet that second requirement without a fixed vocabulary.

**Evaluation rule for this notebook:** per-token loss or perplexity is not comparable across different tokenizers, because a "token" means something different in each. Compare byte-level and token-based models at matched compute and data, and report a tokenizer-independent measure such as bits per byte.

**Go deeper**

*   [Stanford CS336: Language Modeling from Scratch](https://cs336.stanford.edu/) — lecture materials, including a from-scratch BPE implementation
*   [Andrej Karpathy, "Let's build the GPT Tokenizer"](https://www.youtube.com/watch?v=zduSFxRajkE) — a two-hour build of the GPT tokenizer, including its quirks

## Pre-training Data Representation for Transformer Input
Now that we have encoded tokens that follow each other sequentially, we can input them into a neural network to be trained to identify statistical patterns. A language model takes text as a one-dimensional sequence of token IDs from a finite vocabulary, so the representation needs to be efficient.

In the past (before 2017), the state of the art was Recurrent Neural Networks (RNNs) or Long Short Term Memory (LSTM) networks, which process tokens one at a time and carry context forward in a hidden state. That sequential processing "precludes parallelization within training examples" ([Attention Is All You Need](https://arxiv.org/pdf/1706.03762)), which limits how efficiently they can be trained on very large datasets.

In 2017, the seminal paper "Attention is all you need" was published and introduced the Transformer network. The Transformer uses an approach called "self attention": each token can look directly at every earlier token in the context, instead of receiving information step by step through a recurrent state. Attention layers have multiple "heads" that learn different patterns; the original paper reports that many heads "appear to exhibit behavior related to the syntactic and semantic structure of the sentences" ([Attention Is All You Need](https://arxiv.org/pdf/1706.03762)).

*   You can find the paper here: [Attention is All You Need Paper](https://arxiv.org/pdf/1706.03762)


FineWeb was released with 15 trillion tokens; its Hugging Face page now lists more than 18.5 trillion ([FineWeb dataset card](https://huggingface.co/datasets/HuggingFaceFW/fineweb)).

Text chunks can be looked at as atoms of the sequences.

**Call to Action:** Should we add another focus group?

**Output of current step:** At this point we have numbers, but they are only identifiers. There is no context encoded.

**Next Step:** Next we need to add context by feeding these numbers into a neural network.

### What Has Changed in the Transformer Since 2017

Reviewed October 13, 2026. The overview follows Lecture 1 of Stanford CS336 ([course site](https://cs336.stanford.edu/)), with each change linked to its paper.

Modern language models are still Transformers, but most of the parts have been refined since "Attention Is All You Need."

| Component | 2017 original | Common refinement | Source |
|---|---|---|---|
| Activation in the feed-forward layer | ReLU | Gated variants such as SwiGLU | [GLU Variants Improve Transformer](https://arxiv.org/abs/2002.05202) |
| Position information | Fixed sinusoidal encodings added to the input | Rotary position embeddings (RoPE), which build relative position into attention | [RoFormer](https://arxiv.org/abs/2104.09864) |
| Normalization | LayerNorm | RMSNorm: comparable quality while cutting running time by 7% to 64% in its authors' tests | [RMSNorm](https://arxiv.org/abs/1910.07467) |
| Attention cost | Full multi-head attention | Grouped-query attention: several query heads share one key–value head, speeding up generation | [GQA](https://arxiv.org/abs/2305.13245) |
| Feed-forward block | One dense network | Mixture of experts: many expert networks, with each token routed to a few | [Sparsely-gated MoE](https://arxiv.org/abs/1701.06538); [Switch Transformers](https://arxiv.org/abs/2101.03961) |

**Mixture of experts is now the dominant design** for compute-efficient models, according to the lecture. The scale shows why: DeepSeek-V3 has 671 billion total parameters but activates 37 billion for each token ([DeepSeek-V3](https://arxiv.org/abs/2412.19437)), and Kimi K2 has 1 trillion total with 32 billion activated ([Kimi K2](https://arxiv.org/abs/2507.20534)).

**Beyond attention.** Standard attention compares every token with every other, so its cost grows with the square of the sequence length. State-space models and linear-attention variants avoid this. Mamba scales linearly with sequence length and reports five times higher generation throughput than Transformers ([Mamba](https://arxiv.org/abs/2312.00752)), and Gated DeltaNet combines gating with a "delta rule" memory update ([Gated DeltaNet](https://arxiv.org/abs/2412.06464)). In Liang's summary, hybrids of these layers with attention seem to work well.

**Training changes too**

*   **Predicting several tokens at once:** training to predict multiple future tokens improves sample efficiency, and 4-token models were up to three times faster at inference ([Gloeckle et al.](https://arxiv.org/abs/2404.19737)). DeepSeek-V3 uses a multi-token prediction objective ([DeepSeek-V3](https://arxiv.org/abs/2412.19437)).
*   **Optimizers:** AdamW was the default; Muon is increasingly used in recent open models. Kimi K2 used a Muon variant, MuonClip, to pretrain on 15.5 trillion tokens "with zero loss spike" ([Kimi K2](https://arxiv.org/abs/2507.20534)).
*   **Stability:** initialization, learning-rate schedule, and batch size sound like routine settings, but Liang stresses they decide whether a large run reaches state of the art or blows up. Much of training is keeping parameter and gradient sizes in a stable range.
*   **Model shape:** the number of layers, heads, hidden size, and experts looks like a minor choice, but it matters a great deal at scale (see the scaling laws section below).

**Some choices are purely empirical.** The SwiGLU paper ends: "We offer no explanation as to why these architectures seem to work; we attribute their success, as all else, to divine benevolence" ([GLU Variants Improve Transformer](https://arxiv.org/pdf/2002.05202)). Much architecture knowledge comes from running experiments, not from theory.

## Train the Transformer Neural Network

### Updating the Transformer to be an Optimal Model
At this step, we model the statistical relationship of how tokens follow each other in the sequence.

Training consists of matching the encoded tokens with parameters/weights. Since all data is known and concatenated from the encoding step (byte-pair algorithm), the order is known and each following token acts as labeled data.

*   The transformer architecture evaluates all tokens at once to identify following tokens in parallel.
*   Initially the weights are random; because tokens are known and in sequential order, each following token acts as a label, and the preceding tokens are evaluated many times to generalize and predict that label.
*   Training reads the data in batches. After each batch, the optimizer takes one step (an iteration) that adjusts the weights to reduce the loss. Weights start random, so the loss is high at first.
*   One full pass over the dataset is called an epoch. Large language models usually see most of their pretraining data only about once, or a few times, so pretraining is measured in tokens and steps rather than epochs (see [Scaling Laws and Compute-Optimal Training](#scaling-laws)).
*   Over many steps the loss falls, and the model learns patterns that generalize to text it has not seen, ready for inference.
*   The loss indicates how well training is doing.
*   GPUs provide the wide-scale parallelization necessary for modern LLM stacks.
*   Multiple GPUs collaborate to produce the next token.

**Signup page for Transformer Focus Group**
*   [Member Access to Page](https://docs.google.com/document/d/1UfQeuMMEdj0iAEknbFV0YnSyPnNANI9s-8r2-bdGQ6U/edit?usp=sharing)

**Output of current step:** A trained base model ready to accept prompts from users. Using a trained model this way is called inference. Without giving "example usage" this output is not ready to reliably answer questions.

**Next Step:** Base Model Prompting Model Inference from users.

<a id="scaling-laws"></a>

### Scaling Laws and Compute-Optimal Training

Reviewed October 13, 2026, drawing on Lecture 1 of Stanford CS336 ([course site](https://cs336.stanford.edu/)) and the papers linked below.

**Efficiency is the organizing idea.** Rich Sutton's essay "The Bitter Lesson" argues that "general methods that leverage computation are ultimately the most effective, and by a large margin" ([The Bitter Lesson](http://www.incompleteideas.net/IncIdeas/BitterLesson.html)). Liang warns against reading this as "only scale matters, algorithms don't." The better reading is that algorithms that scale are what matter. Model quality comes roughly from efficiency times resources, and efficiency matters more at large scale: when one run can cost hundreds of millions of dollars, even a 5% gain is a big deal. Algorithms do improve: the compute needed to train an image classifier to AlexNet-level accuracy fell 44 times from 2012 to 2019, compared with 11 times from Moore's Law alone ([Hernandez and Brown](https://arxiv.org/abs/2005.04305)).

**Counting compute.** A standard estimate is about 6 floating-point operations per parameter per training token: roughly 2 for the forward pass and twice that for the backward pass ([Kaplan et al.](https://arxiv.org/abs/2001.08361)). Training compute is therefore about 6 × parameters × tokens. For a 70-billion-parameter model trained on 1.4 trillion tokens, that is about 6 × (7 × 10¹⁰) × (1.4 × 10¹²) ≈ 5.9 × 10²³ operations.

**Scaling laws.** Loss falls as a power law in model size, dataset size, and training compute, with some trends spanning more than seven orders of magnitude ([Kaplan et al.](https://arxiv.org/abs/2001.08361)). This means small experiments can predict the outcome of a large one.

**Compute-optimal training.** For a fixed compute budget, model size and training tokens should grow together: "for every doubling of model size the number of training tokens should also be doubled" ([Chinchilla](https://arxiv.org/abs/2203.15556)). Chinchilla, 70 billion parameters trained on 1.4 trillion tokens, used the same compute as the 280-billion-parameter Gopher, trained on 300 billion tokens, and outperformed it. A rough rule of thumb follows: about 20 training tokens per parameter.

**Deliberately training past that point.** The compute-optimal size ignores the cost of running the model afterward. Smaller models are cheaper to serve, so many are trained on far more data. Meta reported that the Chinchilla-optimal budget for an 8-billion-parameter model corresponds to about 200 billion tokens, yet its Llama 3 8B and 70B models kept improving log-linearly up to 15 trillion tokens ([Meta](https://ai.meta.com/blog/meta-llama-3/)).

**Repeating data.** Large-model pretraining usually makes about one pass, or a few, over most of its data. When data is limited, training for up to 4 epochs on repeated data "yields negligible changes to loss compared to having unique data," but with more repetition, the value of extra compute eventually falls to zero ([Muennighoff et al.](https://arxiv.org/abs/2305.16264)).

**Scaling laws are built, not found.** Liang stresses that scaling laws "are not laws of nature." They come from a carefully designed scaling recipe: a mapping from a compute budget to a full set of hyperparameters, such as how the learning rate and batch size should change as scale grows. The goal is hyperparameter transfer, where settings tuned on small models stay correct, or change predictably, at large scale. In his framing, predictability matters at least as much as optimality, because a frontier lab gets to run its largest model only once. OpenAI reported predicting some aspects of GPT-4's performance from models trained with no more than one-thousandth of its compute ([GPT-4 Technical Report](https://arxiv.org/abs/2303.08774)). Marin preregisters its experiments, including scaling-law predictions made before a large run finishes ([Marin](https://marin.community/)).

### Quantum Angle: Training Variational Circuits and Testing Advantage Claims

Quantum callout: a variational quantum circuit uses parameterized gates with a classical optimization loop, which invites comparison with neural-network training without making the two models equivalent ([barren-plateau review](https://arxiv.org/abs/2405.00781)).

Three issues to evaluate before claiming a practical advantage:

1.  **Barren plateaus are conditional.** For affected circuit and cost-function families, gradient concentration can make optimization exponentially difficult; architecture, initialization, observables, loss, and noise matter, so not every variational circuit exhibits the same scaling ([arXiv:2405.00781](https://arxiv.org/abs/2405.00781)).
2.  **Dequantization depends on access assumptions.** Tang's recommendation-system result supplies a classical algorithm under analogous sample/query data access, challenging an exponential speedup claim for that problem rather than disproving all quantum machine learning ([arXiv:1807.04271](https://arxiv.org/abs/1807.04271)).
3.  **Simulation and surrogation have specific scopes.** A December 2025 preprint identifies sufficient conditions for efficient classical simulation or surrogation in studied variational supervised-learning model families; circuit layout, encoding, depth, and non-Clifford resources matter, and these are not universal necessary-and-sufficient tests for quantum advantage ([arXiv:2512.15661](https://arxiv.org/pdf/2512.15661)).

**Avoid the opposite overclaim:** quantum-origin inputs are not the only theoretical setting worth studying; a January 2026 paper establishes a conditional separation for learning quantum observables from classical measurement data, not a general commercial advantage or an LLM-training result ([npj Quantum Information](https://www.nature.com/articles/s41534-025-01162-2)).

## Model Inference
When you send prompts to ChatGPT, Claude, or any other LLM, you are performing inference. Inference predicts one token at a time based on the token window.

*   The parameters (weights) are fixed at this point — no further training.
*   Inference samples from the model’s output distributions (like a biased coin flip) to choose the next token.
*   GPT-2 was one of the first times the full stack came together: Transformer with 1.5 billion parameters (1,542M in the paper), maximum context length of 1024 tokens, trained on WebText: about 40 GB of text from just over 8 million web documents ([GPT-2 paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)).
*   The context window is the set of tokens used to predict the next token.

**What is a Base Model?**
The base model is a token simulator with frozen weights. We can interact with it using:
*   [Hyperbolic](https://app.hyperbolic.ai/models/llama31-405b-base-bf-16)


We can prompt the base model with a few example question-and-answer pairs (few-shot prompting) so that it behaves more like a post-trained model.
- The model will continue the pattern of questions and answers

**Signup page for Base Model Utility Focus Group**
*   [Member Access to Page](https://docs.google.com/document/d/10iPyGPVh7cKhKcjOBIp1gxRnah3_4wA9Ok1QPvTWBgI/edit?usp=sharing)

**Output of this step:** We have a functional base model that is able to predict the next token, but needs context to be useful. Limited context can be provided through few shot question and answer pairs that the base model will then use as context to continue the conversation.

**Next Step:** Provide data to add additional training to a base model which will create a useable chat service.

### Making Inference Fast

Reviewed October 13, 2026, drawing on Lecture 1 of Stanford CS336 ([course site](https://cs336.stanford.edu/)) and the papers linked below.

Inference is growing in importance. It serves chat, but it is also how reinforcement learning generates practice attempts, how test-time "thinking" spends compute, how synthetic data is produced, and how models are evaluated.

**Two phases**

*   **Prefill:** the model reads the whole prompt in parallel and stores intermediate results for every token (the key–value cache, or KV cache). This looks much like training.
*   **Decode:** the model generates one token at a time. Each new token requires reading the model's weights and the growing cache from memory, so the hardware spends most of its time waiting on memory rather than computing. This is why generation is slow and expensive.

**Why memory is the bottleneck.** On a GPU, memory and compute sit in different places, and data has to move between them. Roofline analysis tells you whether a given calculation is limited by compute or by memory bandwidth; for language models it is usually memory. Moving data between GPUs is slower still, so large models depend on careful splitting of parameters, activations, and optimizer state across many chips.

**Ways to speed it up**

| Technique | Idea | Example result (authors' claims) |
|---|---|---|
| Smaller or sparser models | Prune, distill a large model into a small one, or activate only some experts per token | [Distillation](https://arxiv.org/abs/1503.02531) compresses an ensemble's knowledge into one model |
| Quantization | Store weights in fewer bits | [GPTQ](https://arxiv.org/abs/2210.17323) quantizes 175-billion-parameter models in about four GPU hours |
| Smaller KV cache | Share key–value heads across query heads | [Grouped-query attention](https://arxiv.org/abs/2305.13245) |
| Speculative decoding | A cheap draft model guesses several tokens; the full model checks them all in one parallel pass and keeps the ones it agrees with | 2–3 times faster on T5-XXL with identical outputs ([Leviathan et al.](https://arxiv.org/abs/2211.17192)); 2–2.5 times on Chinchilla ([Chen et al.](https://arxiv.org/abs/2302.01318)) |
| Serving systems | Batch requests that arrive at different times, and manage cache memory efficiently | [vLLM](https://arxiv.org/abs/2309.06180) improves throughput 2–4 times at the same latency |

Batching is a serving problem that training does not have: in training, batches are defined in advance, while a live service receives requests at unpredictable times.

**Go deeper:** [How to Scale Your Model](https://jax-ml.github.io/scaling-book/), a free book from Google DeepMind on rooflines, Transformer math, and parallelism, with a chapter on GPUs.

### Quantum Angle: AI-Assisted Quantum Operations

Quantum callout: **AI for quantum has concrete research results and vendor-reported deployments.** Distinguish those evidence levels rather than describing every neural decoder or calibration model as a production real-time component ([AlphaQubit](https://www.nature.com/articles/s41586-024-08148-8); [NVIDIA Ising](https://investor.nvidia.com/news/press-release-details/2026/NVIDIA-Launches-Ising-the-Worlds-First-Open-AI-Models-to-Accelerate-the-Path-to-Useful-Quantum-Computers/default.aspx)).

*   **Peer-reviewed research: AlphaQubit.** The November 2024 paper reports stronger decoding accuracy on the studied Sycamore distance-3 and distance-5 experimental data and additional simulations, not an operational real-time decoder ([Nature](https://www.nature.com/articles/s41586-024-08148-8)). Its discussion identifies an inference-throughput gap relative to approximately 1 μs rounds for superconducting systems; that estimate explicitly excludes latency, which is a separate constraint ([Nature](https://www.nature.com/articles/s41586-024-08148-8)).
*   **Vendor announcement: NVIDIA Ising.** NVIDIA's April 2026 release announces open calibration and decoding models, names adopters, and claims up to 2.5× faster and 3× more accurate decoding than PyMatching in its benchmarks; these are vendor-reported comparisons, not universal performance guarantees ([NVIDIA](https://investor.nvidia.com/news/press-release-details/2026/NVIDIA-Launches-Ising-the-Worlds-First-Open-AI-Models-to-Accelerate-the-Path-to-Useful-Quantum-Computers/default.aspx)).

**Practical evaluation:** report accuracy, inference throughput, end-to-end latency, hardware, and code distance separately. A good offline result is not sufficient evidence of readiness for a real-time control loop.

<a id="post-training---supervised-fine-tuning"></a>

# Post Training — Supervised Fine Tuning

To create a post-trained model we need datasets of conversations so the trained neural network can respond to prompts:
*   Human labelers provide the ideal assistant response for each prompt.
*   Post-training continues training the model from its pre-trained state.

Everything is represented in tokens. Conversations are turned into token sequences, and we need data structures to govern the actions.

## Reinforcement Learning from Human Feedback
![human_feedback](https://drive.google.com/uc?export=view&id=11-lLSXDECJa6mpyC8jKTM5iO9P00JcAi)

**Seminal Paper:** [Training language models to follow instructions with human feedback](https://arxiv.org/pdf/2203.02155) (Ouyang et al., 2022, the InstructGPT paper)
- Hired a team of about 40 contractors on Upwork and through Scale AI to write prompts, demonstrate ideal answers, and rank model outputs.
- Labelers followed detailed written instructions that evolved over the project; excerpts and the full instructions appear in the paper's appendices.

**Signup page for Post Training Supervised Fine Tuning Focus Group**
*   [Member Access to Page](https://docs.google.com/document/d/1drIbgdNlLKWsm-rB19jXseOQsQleuCYEnAwt7xIvSyU/edit?usp=sharing)

**Output of Current Step:** With SFT, results of prompting closely mimic the way a human would respond. Here are a few facts to consider:
- It's All statistics
- Coming from something that is statistically imitating human labelers
- It's a simulation of a human labeler (an expert)
  - asking a question of an expert labeler
- The model will take the pre-trained data and combine it with this post trained data

**Next Step:** Thinking Models (Full Reinforcement Learning)
Full reinforcement learning is not based on directly programing the model with features and what the results should be. Instead, we give the algorithms a goal and allow the neural network to find the optimal solutions on their own.

<a id="post-training---reinforcement-learning"></a>

# Post Training — Reinforcement Learning

With reinforcement learning we teach the LLM to learn from experience through trial and error until it finds good answers.
*   Unlike supervised learning, we do not provide the final answer — we specify the desired result and the system finds an optimal way to achieve it through trial and error.
*   Many solutions are explored in parallel to find good solutions efficiently.

DeepSeek-R1 showed that reinforcement learning with a reward based only on whether final answers are correct, and no human-written reasoning examples, could produce strong reasoning ("Although we do not explicitly teach the model how to reason, it successfully learns improved reasoning strategies through reinforcement learning" ([DeepSeek-R1 paper](https://arxiv.org/pdf/2501.12948)).
- [Deepseek paper](https://arxiv.org/pdf/2501.12948)
- As the model was updated over reinforcement-learning training steps, accuracy improved
- Identified qualitative ways for the model to achieve better results
- Used more tokens to get higher accuracy (longer solutions) - thinking mode
- Focused on chain of thought (better to re-evaluate steps)
- This emerges in the optimization and is not hard coded anywhere.




## Named Methods: PPO, GRPO, and DPO

Reviewed October 13, 2026, drawing on Lecture 1 of Stanford CS336 ([course site](https://cs336.stanford.edu/)) and the papers linked below.

**Why learning from feedback works.** After pretraining and supervised fine-tuning, a model can improve further from weaker signals, because it is often easier to judge a response than to write the ideal one. The basic loop is the same across methods: generate responses, score them with a human, an automatic checker (such as a unit test or a math answer key), or another language model acting as judge, and update the model to prefer the better responses.

| Method | What it does | Where it shows up |
|---|---|---|
| [PPO](https://arxiv.org/abs/1707.06347) — Proximal Policy Optimization, 2017 | A reinforcement-learning algorithm that takes several small, constrained update steps on each batch of sampled responses | InstructGPT, the paper cited above, fine-tunes its model "using the PPO algorithm" against a learned reward model ([InstructGPT](https://arxiv.org/pdf/2203.02155)) |
| [GRPO](https://arxiv.org/abs/2402.03300) — Group Relative Policy Optimization, 2024 | A PPO variant that "foregoes the critic model, instead estimating the baseline from group scores": it samples several answers to the same question and compares each with the group average, saving memory | DeepSeek-R1 uses GRPO as its RL framework ([DeepSeek-R1](https://arxiv.org/abs/2501.12948)) |
| [DPO](https://arxiv.org/abs/2305.18290) — Direct Preference Optimization, 2023 | Skips the separate reward model and reinforcement-learning loop; learns directly from pairs of preferred and rejected responses with "a simple classification loss" | A common, simpler choice when the data is human or model preferences |

**Reinforcement learning is hard to run.** Liang notes that RL algorithms are unstable and hard to tune, and he prefers to stay with supervised training as long as possible. At scale it is also a systems problem: inference servers generate the attempts, sometimes running code in sandboxed environments, while training servers update the model. If the generating workers fall behind, they produce attempts from an outdated version of the model, and teams must constantly balance that against keeping hardware busy.

## American Version of DeepSeek-R1

There has been some concern about using DeepSeek-R1, especially in the US Government. DeepSeek-R1's model weights and code repository are released under the MIT License, which allows modification and derivative works, including distillation ([Hugging Face model card](https://huggingface.co/deepseek-ai/DeepSeek-R1)). Its training data and full training pipeline are not released, so building your own version means fine-tuning or post-training the released weights, or reproducing the method with your own data, rather than re-running DeepSeek's exact training.

**Signup page for Post-Training Reinforcement Learning Focus Group**
*   [Member Access to Page](https://docs.google.com/document/d/1SBTmGHlfN1QDIPlWoXSoRFDIVKOGd7BH7uAiaG9Ewbk/edit?usp=sharing)

**Output of current step:** You now have an end-to-end explanation of how LLMs are built. We hope this makes you curious to build on this knowledge and join one of our focus groups to contribute to this notebook and learn together.

**Next step:** Join a focus group!

<a id="evaluating-language-models"></a>

# Evaluating Language Models

Reviewed October 13, 2026, drawing on Lecture 1 of Stanford CS336 ([course site](https://cs336.stanford.edu/)) and the papers linked below.

Evaluation comes first, not last: it defines the capabilities you want the model to have, which in turn decides what data to collect. Liang separates two purposes that are often confused.

| | Internal metrics | External metrics |
|---|---|---|
| **Used for** | Day-to-day model development | Reporting to users, customers, or reviewers |
| **What matters** | Smooth, predictable behavior across scales, and relative comparisons between experiments | Ecological validity: whether the test reflects real use |
| **Typical example** | Perplexity on held-out text | Task benchmarks and realistic, advanced use cases |

**Perplexity.** Perplexity measures how surprised the model is by held-out text: it is the exponential of the average loss per token, so lower is better. Liang calls it still a very good measure of a model's intrinsic quality, but a perplexity number has little meaning on its own. Because it is measured per token, it cannot be compared across different tokenizers; use bits per byte instead (see [Beyond Tokenizers — Byte-Level Models](#beyond-tokenizers-byte-level-models)).

**Contamination.** Benchmarks published online can end up in training data, which inflates scores. Liang recommends evaluating on data that has never been on the internet where possible.

**Many evaluations, not one number.** Language models are general-purpose, so they need a diverse set of tests. Averaging them into one score is convenient but hides different strengths and weaknesses. Stanford's HELM framework measures 7 metrics (accuracy, calibration, robustness, fairness, bias, toxicity, and efficiency) across 16 core scenarios ([HELM](https://arxiv.org/abs/2211.09110)).

**The metric can create the result.** Some abilities that appear to emerge suddenly at large scale look smooth and predictable under a different metric; one study argues that apparent emergence is often a product of how results are scored ([Schaeffer, Miranda, and Koyejo](https://arxiv.org/abs/2304.15004)). This is one reason internal metrics should change smoothly with scale.

**Questions to ask of any reported result**

1.  Is this an internal development metric or an external, real-use measure?
2.  Could the test data have appeared in training?
3.  Is the comparison at matched compute, data, and tokenizer?
4.  What does the average hide? Look at the individual tests.
5.  How much does the result depend on the choice of metric?

<a id="quantum-computing-and-generative-ai"></a>

# Quantum Computing and Generative AI

**Evidence review: October 3, 2026.** This educational briefing complements the four Quantum Angle callouts above. Dates below identify the cited evidence, not a guarantee that every announcement is the latest available; peer-reviewed results, preprints, vendor reports, roadmaps, drafts, and government directives are distinguished explicitly.

Use three separate questions when evaluating the quantum–AI intersection:

1.  **Quantum for AI:** what learning task, data-access assumptions, and end-to-end baseline support an advantage claim? The reviewed digital-health studies do not show a consistent empirical gain, while narrow theoretical separations do exist ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z); [npj Quantum Information](https://www.nature.com/articles/s41534-025-01162-2)).
2.  **AI for quantum:** is the evidence an offline research result or a documented deployment, and are real-time requirements met? AlphaQubit and NVIDIA Ising illustrate different evidence levels ([Nature](https://www.nature.com/articles/s41586-024-08148-8); [NVIDIA](https://investor.nvidia.com/news/press-release-details/2026/NVIDIA-Launches-Ising-the-Worlds-First-Open-AI-Models-to-Accelerate-the-Path-to-Useful-Quantum-Computers/default.aspx)).
3.  **Quantum and security:** which vulnerable cryptographic dependencies exist, and which authority actually applies to this system? Federal migration obligations differ by scope ([OMB M-26-15](https://www.whitehouse.gov/wp-content/uploads/2026/06/M-26-15-Execution-of-the-Migration-to-Post-Quantum-Cryptography.pdf); [NSA FAQ](https://media.defense.gov/2022/Sep/07/2003071836/-1/-1/0/CSI_CNSA_2.0_FAQ_.PDF)).

The security subsection separates actionable migration planning from predictions about future quantum hardware. It is educational guidance, not a system-specific compliance determination.

**How to read advantage claims:** match the exact observable, depth, precision, and runtime being compared. A March 2026 review describes a classical result at approximately 10⁻¹⁴ error in under ten seconds for a particular five-Trotter-step single-site observable in the 127-qubit kicked-Ising experiment; that is not a blanket result for every depth or observable ([arXiv:2603.18825](https://arxiv.org/html/2603.18825v1)).

## Where the Hardware Actually Is

Treat this as a dated evidence snapshot, not a hardware leaderboard. Physical resources, logical encoding, demonstrated operations, error rates, and post-selection must be read together; the reported Helios and QuEra experiments illustrate why a single logical-qubit count is insufficient ([Quantinuum](https://www.quantinuum.com/blog/introducing-helios-the-most-accurate-quantum-computer-in-the-world); [QuEra](https://www.quera.com/ai-info-page)).

| System or result | Evidence date | Reported figure and scope | Evidence type |
|---|---|---|---|
| Google Willow | Dec 2024 | 105 physical qubits; below-threshold error suppression in the reported surface-code experiment ([Google Research](https://research.google/blog/making-quantum-error-correction-work/)) | Research result described by developer; not a general fault-tolerant application |
| IBM Nighthawk | Nov 2025 | 120 physical qubits and 218 couplers ([IBM](https://newsroom.ibm.com/2025-11-12-ibm-delivers-new-quantum-processors,-software,-and-algorithm-breakthroughs-on-path-to-advantage-and-fault-tolerance)) | Processor announcement |
| IBM qLDPC decoder / Loon effort | Nov 2025 | Classical decoding reported below 480 ns in IBM's fault-tolerance work; this is separate from Nighthawk and is not a demonstrated Nighthawk decoding result ([IBM](https://newsroom.ibm.com/2025-11-12-ibm-delivers-new-quantum-processors,-software,-and-algorithm-breakthroughs-on-path-to-advantage-and-fault-tolerance)) | Vendor-reported decoder result and experimental components |
| Quantinuum Helios | Nov 2025 | Vendor reports 98 physical qubits and 99.921% two-qubit gate fidelity; separately, a 48-logical-qubit error-correction demonstration using 2:1 encoding, not 48 arbitrary fault-tolerant application qubits ([Quantinuum](https://www.quantinuum.com/blog/introducing-helios-the-most-accurate-quantum-computer-in-the-world)) | Product announcement plus code-specific demonstration |
| QuEra research | 2025 result, page reviewed Oct 2026 | A 96-logical-qubit research demonstration, not the specification of a generally available 96-logical-qubit product ([QuEra](https://www.quera.com/ai-info-page)) | Research demonstration described by vendor |
| QuEra Libra | Target 2028, page reviewed Oct 2026 | Target of 256 logical qubits and logical error rate 10⁻⁶; not an achieved capability in the cited source ([QuEra](https://www.quera.com/ai-info-page)) | Roadmap |
| IBM Starling | Target 2029 | Target of 100 million gates on 200 logical qubits ([IBM](https://www.ibm.com/quantum/blog/large-scale-ftqc)) | Roadmap |
| Microsoft Majorana 1 | Feb–Mar 2025 announcement and coverage | Nature's contemporaneous reporting said the accompanying paper did not establish the claimed topological qubits; this row records that historical evidence dispute, not a complete assessment of later work ([Nature](https://www.nature.com/articles/d41586-025-00683-2)) | Contested announcement at that date |

**Evaluation checklist:** request code family and distance, error detection versus correction, accepted-shot fraction, demonstrated gates, logical error rate, and the duration of the experiment. Do not substitute roadmap targets for measured results.

**Independent evaluation effort:** DARPA announced 11 Quantum Benchmarking Initiative Stage B selections on November 6, 2025, with the possibility of additional selections; the initiative assesses whether utility-scale computing, whose computational value exceeds its cost, can be achieved by 2033, rather than guaranteeing delivery by that date ([DARPA](https://www.darpa.mil/research/programs/quantum-benchmarking-initiative/stage-b-selection)).

## Technique Maturity and Evidence

Use these categories to plan a curriculum or evaluation, not to predict a universal adoption date. Availability, research evidence, and commercial usefulness are separate questions.

### Available classical methods and migration work
These items do not require a quantum processor, but they still need task-specific evaluation. PQC is security engineering, not a quantum-computing accelerator.

*   **Tensor networks:** classical simulation and decoding methods provide important baselines; AlphaQubit compares against a tensor-network decoder, while the kicked-Ising review discusses the reach and limitations of classical simulation ([Nature](https://www.nature.com/articles/s41586-024-08148-8); [arXiv:2603.18825](https://arxiv.org/html/2603.18825v1)).
*   **Quantum-inspired linear algebra:** Tang's classical recommendation algorithm is a useful case study in how sample/query access assumptions affect a speedup comparison ([arXiv:1807.04271](https://arxiv.org/abs/1807.04271)).
*   **Post-quantum cryptography:** NIST finalized FIPS 203, 204, and 205 in August 2024; inventory, integration, interoperability testing, and staged migration are actionable without quantum hardware ([NIST](https://www.nist.gov/news-events/news/2024/08/nist-releases-first-3-finalized-post-quantum-encryption-standards); [NCCoE FAQ](https://pages.nist.gov/nccoe-migration-post-quantum-cryptography/FAQ/)).

### Research demonstrations and vendor-reported deployments
*   **AI-assisted operations:** AlphaQubit provides peer-reviewed decoding research, while NVIDIA reports model availability and adopters for Ising; these sources should not be collapsed into a blanket real-time production claim ([Nature](https://www.nature.com/articles/s41586-024-08148-8); [NVIDIA](https://investor.nvidia.com/news/press-release-details/2026/NVIDIA-Launches-Ising-the-Worlds-First-Open-AI-Models-to-Accelerate-the-Path-to-Useful-Quantum-Computers/default.aspx)).
*   **Quantum Echoes:** Google's October 2025 announcement accompanied a Nature publication, not merely a preprint; distinguish its beyond-classical OTOC benchmark claim from its molecular NMR demonstration, which Google explicitly said was not yet beyond classical capability ([Google Research](https://research.google/blog/a-verifiable-quantum-advantage/)).
*   **Learning from classical measurement data:** a 2026 theoretical result establishes an advantage for a constructed quantum-observable learning task under complexity assumptions, not general quantum ML superiority ([npj Quantum Information](https://www.nature.com/articles/s41534-025-01162-2)).

### Roadmaps, resource estimates, and open questions
*   **Larger fault-tolerant systems:** Libra's 2028 and Starling's 2029 targets are vendor roadmaps, not measured capabilities or promised dates for broadly useful applications ([QuEra](https://www.quera.com/ai-info-page); [IBM](https://www.ibm.com/quantum/blog/large-scale-ftqc)).
*   **End-to-end learning advantage:** the cited review of 16 digital-health studies found no clear, consistent empirical benefit; that limited literature does not settle every learning task or establish a practical LLM-training advantage ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z)).
*   **RSA-2048 cryptanalysis:** a 2025 paper estimates a construction requiring fewer than one million noisy qubits under stated hardware and error-correction assumptions; a resource estimate is not an implemented attack, a hardware delivery forecast, or a universal lower bound ([arXiv:2505.15917](https://arxiv.org/abs/2505.15917)).

## Quantum and the Security of AI Systems

AI systems can depend on quantum-vulnerable public-key mechanisms for key establishment, authentication, and signatures, while bulk data encryption commonly uses symmetric algorithms; migration therefore begins by identifying actual dependencies, not by assuming every model file uses the same cryptography ([NIST transition draft](https://nvlpubs.nist.gov/nistpubs/ir/2024/NIST.IR.8547.ipd.pdf)). Government deadlines below apply to specified systems and authorities, not automatically to every AI vendor or private business ([OMB M-26-15](https://www.whitehouse.gov/wp-content/uploads/2026/06/M-26-15-Execution-of-the-Migration-to-Post-Quantum-Cryptography.pdf)).

### The threat, stated precisely
*   **Harvest now, decrypt later:** captured traffic protected through vulnerable public-key key establishment may be exposed by a future cryptographically relevant quantum computer; the risk is especially important for data that must remain confidential for many years, but this is not a claim that all captured ciphertext becomes decryptable ([NIST transition draft](https://nvlpubs.nist.gov/nistpubs/ir/2024/NIST.IR.8547.ipd.pdf)).
*   **Signatures are a separate migration concern:** authenticity and integrity require their own inventory and transition planning, rather than treating signature replacement as an encryption-only task ([OMB M-26-15](https://www.whitehouse.gov/wp-content/uploads/2026/06/M-26-15-Execution-of-the-Migration-to-Post-Quantum-Cryptography.pdf)).
*   **Resource estimates are conditional:** the 2025 RSA-2048 study is an algorithm-and-hardware resource estimate, not evidence that an operational attack exists or will arrive on a specific date ([arXiv:2505.15917](https://arxiv.org/abs/2505.15917)).

### The standards
NIST finalized FIPS 203 (ML-KEM), FIPS 204 (ML-DSA), and FIPS 205 (SLH-DSA) on August 13, 2024; these cover key encapsulation and digital signatures, not wholesale replacement of symmetric bulk encryption ([NIST](https://www.nist.gov/news-events/news/2024/08/nist-releases-first-3-finalized-post-quantum-encryption-standards)). NIST's March 2025 announcement selected HQC for standardization; selection is not itself a finalized standard ([NIST](https://www.nist.gov/news-events/news/2025/03/nist-selects-hqc-fifth-algorithm-post-quantum-encryption)).

The linked NIST IR 8547 is the November 2024 **initial public draft**: it proposes deprecation after 2030 for specified 112-bit-strength public-key uses and disallowance after 2035 for quantum-vulnerable public-key algorithms; do not present this draft as a final rule banning all classical cryptography ([NIST IR 8547 draft](https://nvlpubs.nist.gov/nistpubs/ir/2024/NIST.IR.8547.ipd.pdf)).

### Dates, scope, and authority
| Authority | System scope | Action | Date | Status of cited authority |
|---|---|---|---|---|
| CNSSP 15 as summarized by NSA | New National Security System (NSS) acquisitions | CNSA 2.0 compliance, **unless otherwise noted**; consult applicable profiles and exceptions | Jan 1, 2027 | Policy requirement summarized in [NSA FAQ](https://media.defense.gov/2022/Sep/07/2003071836/-1/-1/0/CSI_CNSA_2.0_FAQ_.PDF) |
| June 2026 executive action, §4(b) | Covered agency high-value assets and high-impact systems, excluding NSS | Migrate key establishment; migrate digital signatures | Dec 31, 2030; Dec 31, 2031, respectively | Executive direction with defined scope ([White House](https://www.whitehouse.gov/presidential-actions/2026/06/securing-the-nation-against-advanced-cryptographic-attacks/)) |
| June 2026 executive action, §6(c) | Contractor coverage to be defined through FAR rulemaking | Directs publication of a **proposed** rule within 180 days, contemplating compliance by Dec 31, 2030 | Proposed-rule timetable; contemplated 2030 compliance date | This directive alone is **not an effective final FAR rule** ([White House](https://www.whitehouse.gov/presidential-actions/2026/06/securing-the-nation-against-advanced-cryptographic-attacks/)) |
| CNSSP 15 as summarized by NSA | NSS, subject to applicable exceptions and profiles | Phase out equipment/services unable to support CNSA 2.0; mandate its use | Dec 31, 2030; Dec 31, 2031, respectively | Distinct NSS policy, not a blanket statement about every DoD system ([NSA FAQ](https://media.defense.gov/2022/Sep/07/2003071836/-1/-1/0/CSI_CNSA_2.0_FAQ_.PDF)) |
| OMB M-26-15 | Covered federal agency systems; NSS expressly excluded | Phased migration, with remaining systems addressed through risk-based planning and commercial availability | 2026–2035 phases | Agency implementation memorandum ([OMB](https://www.whitehouse.gov/wp-content/uploads/2026/06/M-26-15-Execution-of-the-Migration-to-Post-Quantum-Cryptography.pdf)) |
| NIST IR 8547, November 2024 version | Specified quantum-vulnerable public-key algorithms and uses | Proposed deprecation/disallowance transition | After 2030 / after 2035, depending on strength and use | **Initial public draft**, not a final universal deadline ([NIST](https://nvlpubs.nist.gov/nistpubs/ir/2024/NIST.IR.8547.ipd.pdf)) |

OMB M-26-15 describes inventory/strategy/governance in 2026–27, pilots in 2027–28, prioritized key establishment in 2028–30, signatures in 2031, and remaining migration through 2035; read the priority categories and risk/commercial-availability qualifications, rather than applying the dates indiscriminately ([OMB M-26-15](https://www.whitehouse.gov/wp-content/uploads/2026/06/M-26-15-Execution-of-the-Migration-to-Post-Quantum-Cryptography.pdf)). For a specific organization, verify the current policy version, contract clauses, system classification, and applicable acquisition profile before asserting a compliance obligation.

### Practical first step
Build a cryptographic inventory that records algorithms, key establishment, signatures, certificates, libraries, owners, data lifetimes, and vendor dependencies; inventory is a starting point, not a certification of readiness, and must lead to risk prioritization and interoperability testing ([NCCoE migration FAQ](https://pages.nist.gov/nccoe-migration-post-quantum-cryptography/FAQ/)).

**Quantum and AI Focus Group:** planned; registration is not yet available. Proposed scope: maintain the evidence snapshot, track scoped PQC requirements, and compare quantum claims against appropriate classical baselines.

## Suggested Hands-On Additions (Future Breakout Notebook)

**Status: proposed, not implemented or executed in this notebook.** These are designs for a future breakout, not reproducible results, validated runtimes, or claims of quantum advantage; each will need code, dependencies, seeds, baselines, and acceptance criteria.

1.  **Compare input representations:** tokenize a sentence and separately prepare basis, angle, and amplitude encodings of a derived feature vector; report preprocessing, state-preparation, measurement, and repetition costs without treating these representations as equivalent.
2.  **Study gradient concentration:** vary qubit count, ansatz depth, observable locality, initialization, and noise across multiple seeds; estimate gradient variance with uncertainty rather than promising that two small classifiers will demonstrate asymptotic barren-plateau behavior ([review](https://arxiv.org/abs/2405.00781)).
3.  **Test a classical surrogate:** compare a shallow circuit with a surrogate on held-out inputs and report sampling and training costs; fitting one surrogate is not proof that all circuits are efficiently simulable ([arXiv:2512.15661](https://arxiv.org/pdf/2512.15661)).
4.  **Explore tensor-network compression:** replace a small model's dense layer with a matrix-product-operator factorization and measure parameter count, memory, runtime, and accuracy against a matched baseline.
5.  **Prototype a neural decoder:** train on synthetic syndrome data, then report logical error rates, inference throughput, and latency separately; do not describe the exercise as a reproduction of AlphaQubit without matching its protocol ([Nature](https://www.nature.com/articles/s41586-024-08148-8)).
6.  **Build a crypto inventory:** combine dependency scanning with configuration, runtime, certificate, and vendor review; record unknowns rather than assigning a blanket 'PQC-ready' label, and add interoperability and migration planning as follow-on work ([NCCoE FAQ](https://pages.nist.gov/nccoe-migration-post-quantum-cryptography/FAQ/)).

**Proposed toolchains:** evaluate Qiskit, PennyLane, or CUDA-Q for circuit exercises and a tensor-network library for compression. Cryptography reference implementations may be used for education, but this proposal does not endorse them for production deployment or establish compliance.

<a id="further-learning"></a>

# Further Learning

Free resources for going deeper than this notebook, reviewed October 13, 2026.

*   [Stanford CS336: Language Modeling from Scratch](https://cs336.stanford.edu/) — Percy Liang and colleagues' course on building a language model end to end: tokenization, architecture, GPU systems, scaling laws, data, and alignment. Lecture materials are posted on the site.
*   [How to Scale Your Model](https://jax-ml.github.io/scaling-book/) — a free Google DeepMind book on rooflines, Transformer math, and parallelism for training and inference, including a chapter on GPUs.
*   [Andrej Karpathy, "Let's build the GPT Tokenizer"](https://www.youtube.com/watch?v=zduSFxRajkE) — a two-hour build of the GPT tokenizer, including its quirks.
*   [HELM](https://crfm.stanford.edu/helm/) — Stanford's holistic evaluation framework and leaderboards.
*   [OLMo](https://allenai.org/olmo) — AI2's fully open models, with training data and code.
*   [Marin](https://marin.community/) and its [GitHub repository](https://github.com/marin-community/marin) — open development of foundation models, with experiments and training runs visible as they happen.

<a id="glossary"></a>

# Glossary
A brief definition of terms found in this notebook to aid understanding. Each entry includes a short definition and a Perplexity link for additional context.

### Artificial Intelligence (AI)
The field of computer science focused on building systems that can perform tasks that typically require human intelligence (e.g., reasoning, learning, perception). [Perplexity Definition](https://www.perplexity.ai/search?q=artificial+intelligence)

### Base model
A pre-trained language model with frozen weights that predicts the next token in a sequence. It has not been post-trained for chat or instruction-following; useful for inference experiments and as the starting point for SFT or RL. [Perplexity Definition](https://www.perplexity.ai/search?q=base+language+model+LLM)

### Byte-level (token-free) model
A language model that reads raw bytes instead of tokens from a fixed vocabulary, removing the separate tokenizer. Research models such as ByT5, MEGABYTE, Byte Latent Transformer, and H-Net group bytes into patches or learned chunks to keep sequences manageable; as of 2026, frontier models still use tokenizers. [Perplexity Definition](https://www.perplexity.ai/search?q=byte-level+token-free+language+model)

### Compute-optimal training
Choosing model size and number of training tokens to get the lowest loss for a fixed compute budget. The Chinchilla study found both should grow together, roughly 20 training tokens per parameter; many models are deliberately trained on more data so they are cheaper to run. [Perplexity Definition](https://www.perplexity.ai/search?q=compute-optimal+training+Chinchilla)

### Context window
The set of tokens (e.g., the current prompt plus prior turns) that the model uses to predict the next token. Larger context windows allow longer conversations and documents. [Perplexity Definition](https://www.perplexity.ai/search?q=context+window+LLM)

### Data contamination
When benchmark or test data appears in a model's training data, inflating its evaluation scores. [Perplexity Definition](https://www.perplexity.ai/search?q=data+contamination+LLM+benchmark)

### Direct Preference Optimization (DPO)
A post-training method that learns directly from pairs of preferred and rejected responses with a simple classification loss, without a separate reward model or reinforcement-learning loop. [Perplexity Definition](https://www.perplexity.ai/search?q=direct+preference+optimization+DPO)

### Epoch
One full pass over the training dataset. Smaller models are often trained for many epochs; large language model pretraining usually passes over most of its data only about once, or a few times, because repeated data gives diminishing returns. [Perplexity Definition](https://www.perplexity.ai/search?q=epoch+machine+learning)

### Generative Artificial Intelligence (GenAI)
AI systems that generate new content (text, images, code, etc.) rather than only classifying or retrieving existing data. LLMs are a prominent form of GenAI. [Perplexity Definition](https://www.perplexity.ai/search?q=generative+artificial+intelligence)

### Group Relative Policy Optimization (GRPO)
A variant of PPO that samples several answers to the same question and scores each against the group average, removing the separate value (critic) model. Used to train DeepSeek-R1. [Perplexity Definition](https://www.perplexity.ai/search?q=GRPO+group+relative+policy+optimization)

### Inference
Using a trained model to produce outputs (e.g., generating the next token or a full response) without updating its weights. What happens when you send a prompt to ChatGPT or Claude. [Perplexity Definition](https://www.perplexity.ai/search?q=inference+machine+learning)

### KV cache
The stored key and value vectors for tokens already processed, so the model does not recompute them for each new token. Its size grows with context length and is a major memory cost during inference. [Perplexity Definition](https://www.perplexity.ai/search?q=KV+cache+LLM+inference)

### Large Language Model (LLM)
A language model trained on very large amounts of text (and sometimes other data) with many parameters, typically based on transformer architectures and capable of general-purpose text generation and understanding. [Perplexity Definition](https://www.perplexity.ai/search?q=large+language+model)

### Loss
A measure of how wrong the model’s predictions are during training. Training aims to minimize loss so that predictions match the data. [Perplexity Definition](https://www.perplexity.ai/search?q=loss+function+machine+learning)

### Mixture of experts (MoE)
An architecture in which the feed-forward block contains many expert networks and each token is routed to only a few, so total parameters can grow much faster than the compute per token. [Perplexity Definition](https://www.perplexity.ai/search?q=mixture+of+experts+LLM)

### Perplexity (evaluation metric)
A measure of how well a model predicts held-out text: the exponential of the average loss per token, so lower is better. Not comparable across different tokenizers. Unrelated to Perplexity.ai, the search product. [Perplexity Definition](https://www.perplexity.ai/search?q=perplexity+language+model+metric)

### Reinforcement Learning from Human Feedback (RLHF)
Post-training where humans (or other models) rate or rank model outputs; the model is then updated to favor higher-rated responses. Used to align behavior with human preferences. [Perplexity Definition](https://www.perplexity.ai/search?q=RLHF+reinforcement+learning+human+feedback)

### Scaling law
An empirical relationship, usually a power law, between a model's loss and its size, training data, or compute. Used to predict large-model results from small experiments. [Perplexity Definition](https://www.perplexity.ai/search?q=scaling+laws+neural+language+models)

### Speculative decoding
An inference speed-up in which a small draft model proposes several tokens and the full model checks them in one parallel pass, keeping those it agrees with. [Perplexity Definition](https://www.perplexity.ai/search?q=speculative+decoding+LLM)

### Supervised Fine Tuning (SFT)
Post-training that continues from a pre-trained model using labeled prompt–response pairs (e.g., human-written ideal answers), teaching the model to follow instructions or mimic a specific style. [Perplexity Definition](https://www.perplexity.ai/search?q=supervised+fine+tuning+LLM)

### Tokenization
Converting raw text into a sequence of tokens (subword or word units) that the model can process. Methods like byte-pair encoding (BPE) balance vocabulary size and sequence length. [Perplexity Definition](https://www.perplexity.ai/search?q=tokenization+LLM+NLP)

### Transformer
A neural network architecture that uses self-attention to process all tokens in a sequence in parallel (rather than strictly left-to-right). The basis for most modern LLMs; introduced in “Attention is all you need.” [Perplexity Definition](https://www.perplexity.ai/search?q=transformer+neural+network)

### Barren plateau
A gradient-concentration phenomenon that can make optimization exponentially difficult in affected variational circuit families; its presence depends on circuit, cost-function, initialization, and noise assumptions ([arXiv:2405.00781](https://arxiv.org/abs/2405.00781)). [Perplexity Definition](https://www.perplexity.ai/search?q=barren+plateau+quantum+machine+learning)

### Cryptographically relevant quantum computer (CRQC)
A quantum computer with sufficient capability to attack deployed quantum-vulnerable cryptography; this definition does not specify an arrival date or mean that all cryptographic algorithms would be broken ([NIST transition draft](https://nvlpubs.nist.gov/nistpubs/ir/2024/NIST.IR.8547.ipd.pdf)). [Perplexity Definition](https://www.perplexity.ai/search?q=cryptographically+relevant+quantum+computer)

### Dequantization
Developing a classical counterpart that challenges a proposed quantum speedup under specified data-access and approximation assumptions; Tang's recommendation-system result is an important example, not a disproof of all quantum advantage ([arXiv:1807.04271](https://arxiv.org/abs/1807.04271)). [Perplexity Definition](https://www.perplexity.ai/search?q=dequantization+quantum+algorithms+Ewin+Tang)

### Harvest now, decrypt later (HNDL)
Collecting encrypted traffic for possible later decryption using a future CRQC, particularly where vulnerable public-key key establishment protects long-lived confidential data; not all captured ciphertext has the same exposure ([NIST transition draft](https://nvlpubs.nist.gov/nistpubs/ir/2024/NIST.IR.8547.ipd.pdf)). [Perplexity Definition](https://www.perplexity.ai/search?q=harvest+now+decrypt+later)

### Logical qubit
A qubit encoded within a quantum error-correcting code; an experiment may demonstrate error detection, correction, or a restricted set of logical operations, so the count alone does not specify fault-tolerant application capability ([Quantinuum's distinct Helios demonstrations](https://www.quantinuum.com/blog/introducing-helios-the-most-accurate-quantum-computer-in-the-world)). [Perplexity Definition](https://www.perplexity.ai/search?q=logical+qubit+quantum+error+correction)

### Post-quantum cryptography (PQC)
Classical algorithms designed to resist quantum attack. The first three standards, FIPS 203/204/205, were published August 13, 2024 ([NIST](https://www.nist.gov/news-events/news/2024/08/nist-releases-first-3-finalized-post-quantum-encryption-standards)). Not a quantum technology. [Perplexity Definition](https://www.perplexity.ai/search?q=post-quantum+cryptography+FIPS+203+204+205)

### Quantum machine learning (QML)
Machine-learning methods using quantum computational resources, including but not limited to variational circuits and quantum kernels; evidence of advantage is task- and assumption-specific ([npj Digital Medicine](https://www.nature.com/articles/s41746-025-01597-z); [npj Quantum Information](https://www.nature.com/articles/s41534-025-01162-2)). [Perplexity Definition](https://www.perplexity.ai/search?q=quantum+machine+learning)

### Shor's algorithm
Quantum algorithms for integer factoring and discrete logarithms: factoring threatens RSA, while the discrete-logarithm capability threatens finite-field Diffie–Hellman and elliptic-curve cryptography, given a sufficiently capable fault-tolerant quantum computer ([NIST transition draft](https://nvlpubs.nist.gov/nistpubs/ir/2024/NIST.IR.8547.ipd.pdf)). [Perplexity Definition](https://www.perplexity.ai/search?q=Shor%27s+algorithm)

### Tensor network
A representation of a high-dimensional object as connected lower-order tensors; tensor-network methods provide classical simulation and decoding baselines whose efficiency depends on the problem's structure ([arXiv:2603.18825](https://arxiv.org/html/2603.18825v1); [AlphaQubit](https://www.nature.com/articles/s41586-024-08148-8)). [Perplexity Definition](https://www.perplexity.ai/search?q=tensor+network+machine+learning)

### Variational quantum circuit (VQC)
A parameterized quantum circuit whose parameters are adjusted using a classical optimization loop; trainability depends on the circuit and objective, rather than following automatically from the analogy to neural networks ([barren-plateau review](https://arxiv.org/abs/2405.00781)). [Perplexity Definition](https://www.perplexity.ai/search?q=variational+quantum+circuit)